# Proyecto Centinela · Fase 2 — Rama A: CNN con Transfer Learning
### Matrices de recurrencia · ResNet18 vs EfficientNet-B0 · Grad-CAM

**Autores:** Estefanny Ruíz González · Miguel Alarcón Ojeda
**Curso:** Profundización I — Redes Neuronales / Deep Learning (EA-N-F-004)
**Maestría en Ciencia de Datos — Universidad Santo Tomás**

---

**Problema:** clasificar el comportamiento de la TRM en 30 días en tres categorías:
depreciación, estabilidad o apreciación, usando matrices de recurrencia como imágenes.

**Novedad respecto a la versión anterior:**
- Se comparan **dos arquitecturas preentrenadas**: ResNet18 y EfficientNet-B0.
- Se agrega **data augmentation**: rotación, flip horizontal y variación de brillo.
- La decisión Feature Extraction vs Fine-Tuning se justifica con evidencia empírica.


## 0. Preparación del entorno

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (confusion_matrix, classification_report,
                              ConfusionMatrixDisplay)
import time

SEMILLA = 42
WINDOW  = 30
BATCH   = 32
EPOCAS  = 20
LR      = 0.001

np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Dispositivo:", device)


## 1. Carga de datos y etiquetas

In [ ]:
from google.colab import files
subido = files.upload()  # subir trm_diaria_limpia.csv


In [ ]:
df = pd.read_csv("trm_diaria_limpia.csv", parse_dates=["fecha"])
df = df.sort_values("fecha").reset_index(drop=True)
df["retorno"] = df["trm"].pct_change() * 100
df = df.dropna(subset=["retorno"]).reset_index(drop=True)

retornos = df["retorno"].values
nombres_clases = ["Depreciación", "Estable", "Apreciación"]

etiquetas = []
for i in range(WINDOW, len(retornos)):
    acum = retornos[i-WINDOW:i].sum()
    if   acum >  1.0: etiquetas.append(0)
    elif acum < -1.0: etiquetas.append(2)
    else:             etiquetas.append(1)
etiquetas = np.array(etiquetas)

print("Ventanas disponibles:", len(etiquetas))
for c, n in enumerate(nombres_clases):
    cnt = (etiquetas==c).sum()
    print(f"  Clase {c} — {n}: {cnt} ({cnt/len(etiquetas)*100:.1f}%)")


## 2. Construcción de matrices de recurrencia

Cada ventana de 30 días de retornos genera una imagen de 30×30 píxeles
donde el pixel (i,j) vale 1 si |retorno_i − retorno_j| < 0.5%, y 0 si no.


In [ ]:
def matriz_recurrencia(serie, umbral=0.5):
    n = len(serie)
    mat = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        for j in range(n):
            mat[i,j] = 1.0 if abs(serie[i]-serie[j]) < umbral else 0.0
    return mat

print("Pre-calculando matrices de recurrencia...")
todas_matrices = np.zeros((len(etiquetas), WINDOW, WINDOW), dtype=np.float32)
for i in range(len(etiquetas)):
    todas_matrices[i] = matriz_recurrencia(retornos[i:i+WINDOW])
    if (i+1) % 1500 == 0:
        print(f"  {i+1}/{len(etiquetas)} procesadas...")
print(f"Listo: {todas_matrices.shape} · {todas_matrices.nbytes/1e6:.1f} MB")


## 3. Partición, data augmentation y Dataset

### Data augmentation
Se aplica a las imágenes de **entrenamiento únicamente**. Para validación
y test solo se redimensiona y normaliza (sin augmentation) para evaluar
sobre condiciones reales. Las transformaciones elegidas son:

- **Rotación aleatoria ±15°:** la orientación de los patrones en la matriz
  puede variar; esto hace al modelo más robusto.
- **Flip horizontal:** los patrones de recurrencia son a veces simétricos.
- **Variación de brillo (±20%):** simula variaciones en la intensidad de
  los retornos sin cambiar la estructura temporal.


In [ ]:
# Partición cronológica 70/15/15
indices = np.arange(len(etiquetas))
n = len(indices)
i_tr, i_va = int(n*0.70), int(n*0.85)
idx_train = indices[:i_tr]
idx_val   = indices[i_tr:i_va]
idx_test  = indices[i_va:]

print(f"Train: {len(idx_train)} | Val: {len(idx_val)} | Test: {len(idx_test)}")

# Transform CON augmentation (solo train)
transform_train = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Transform SIN augmentation (val y test)
transform_eval = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


In [ ]:
class TRMRecurrenceDataset(Dataset):
    def __init__(self, matrices, etiquetas, indices, transform=None):
        self.matrices  = matrices
        self.etiquetas = etiquetas
        self.indices   = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        i = self.indices[idx]
        mat = self.matrices[i]
        # Convertir a uint8 RGB para PIL
        img_rgb = (np.stack([mat]*3, axis=-1) * 255).astype(np.uint8)
        if self.transform:
            img = self.transform(img_rgb)
        else:
            img = torch.tensor(mat).unsqueeze(0).repeat(3,1,1).float()
        return img, int(self.etiquetas[idx])

ds_train = TRMRecurrenceDataset(todas_matrices, etiquetas, idx_train, transform_train)
ds_val   = TRMRecurrenceDataset(todas_matrices, etiquetas, idx_val,   transform_eval)
ds_test  = TRMRecurrenceDataset(todas_matrices, etiquetas, idx_test,  transform_eval)

dl_train = DataLoader(ds_train, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
dl_val   = DataLoader(ds_val,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
dl_test  = DataLoader(ds_test,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

print(f"Batches train: {len(dl_train)} | val: {len(dl_val)} | test: {len(dl_test)}")


In [ ]:
# Mostrar ejemplos de augmentation
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for col in range(4):
    idx = idx_train[col * 300]
    mat = todas_matrices[idx]
    img_rgb = (np.stack([mat]*3, axis=-1) * 255).astype(np.uint8)
    # Original
    axes[0][col].imshow(mat, cmap='binary', origin='upper')
    axes[0][col].set_title(f"Original\n{nombres_clases[etiquetas[idx]]}", fontsize=8)
    axes[0][col].axis('off')
    # Con augmentation
    from PIL import Image
    pil = Image.fromarray(img_rgb)
    aug = transforms.Compose([
        transforms.RandomRotation(15),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.2)
    ])(pil)
    axes[1][col].imshow(aug, origin='upper')
    axes[1][col].set_title("Con augmentation", fontsize=8)
    axes[1][col].axis('off')

plt.suptitle("Data augmentation — efecto visual", fontsize=11)
plt.tight_layout()
plt.show()


## 4. Selección de arquitectura: ResNet18 vs EfficientNet-B0

La rúbrica exige comparar al menos dos arquitecturas preentrenadas.
Se evalúan ResNet18 y EfficientNet-B0 bajo las mismas condiciones.

| Arquitectura | Parámetros | Profundidad | Top-1 ImageNet | Estrategia |
|---|---|---|---|---|
| ResNet18 | 11.2 M | 18 capas | 69.8% | Feature Extraction |
| EfficientNet-B0 | 5.3 M | 82 capas (escalado) | 77.1% | Feature Extraction |

**Feature Extraction vs Fine-Tuning:**
Con solo 6.277 imágenes de 30×30 píxeles (dominio muy diferente a ImageNet),
el Fine-Tuning profundo arriesga sobreajustar y destruir representaciones
preentrenadas útiles. Feature Extraction es la estrategia apropiada:
se congela todo excepto la capa final, aprovechando los detectores
de bajo nivel (bordes, texturas) que sí son transferibles.


In [ ]:
def crear_modelo(tipo="resnet18"):
    if tipo == "resnet18":
        modelo = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        for param in modelo.parameters():
            param.requires_grad = False
        modelo.fc = nn.Linear(512, 3)
        n_params = sum(p.numel() for p in modelo.parameters())
        n_train  = sum(p.numel() for p in modelo.parameters() if p.requires_grad)

    elif tipo == "efficientnet_b0":
        modelo = models.efficientnet_b0(
            weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        for param in modelo.parameters():
            param.requires_grad = False
        modelo.classifier[1] = nn.Linear(
            modelo.classifier[1].in_features, 3)
        n_params = sum(p.numel() for p in modelo.parameters())
        n_train  = sum(p.numel() for p in modelo.parameters() if p.requires_grad)

    print(f"{tipo}: {n_params:,} totales | {n_train:,} entrenables "
          f"({n_train/n_params*100:.2f}%)")
    return modelo.to(device)

m_resnet = crear_modelo("resnet18")
m_effnet = crear_modelo("efficientnet_b0")


## 5. Entrenamiento de ambas arquitecturas

In [ ]:
def entrenar(modelo, dl_tr, dl_va, nombre, epocas=EPOCAS, lr=LR):
    criterio = nn.CrossEntropyLoss()
    optim = torch.optim.Adam(
        [p for p in modelo.parameters() if p.requires_grad], lr=lr)

    hist_tr, hist_va = [], []
    mejor_val   = float("inf")
    mejor_pesos = None
    t0 = time.time()

    for epoca in range(epocas):
        modelo.train()
        loss_tr = 0.0
        for imgs, labels in dl_tr:
            imgs, labels = imgs.to(device), labels.to(device)
            optim.zero_grad()
            out  = modelo(imgs)
            loss = criterio(out, labels)
            loss.backward()
            optim.step()
            loss_tr += loss.item()
        loss_tr /= len(dl_tr)

        modelo.eval()
        loss_va = 0.0
        with torch.no_grad():
            for imgs, labels in dl_va:
                out = modelo(imgs.to(device))
                loss_va += criterio(out, labels.to(device)).item()
        loss_va /= len(dl_va)

        hist_tr.append(loss_tr)
        hist_va.append(loss_va)

        if loss_va < mejor_val:
            mejor_val   = loss_va
            mejor_pesos = {k: v.clone() for k,v in modelo.state_dict().items()}

        if epoca % 5 == 0:
            print(f"  [{nombre}] Época {epoca:2d} | "
                  f"train: {loss_tr:.4f} | val: {loss_va:.4f}")

    modelo.load_state_dict(mejor_pesos)
    elapsed = time.time() - t0
    print(f"  [{nombre}] Mejor val: {mejor_val:.4f} | "
          f"Tiempo: {elapsed/60:.1f} min")
    return hist_tr, hist_va

print("=== Entrenando ResNet18 ===")
torch.manual_seed(SEMILLA)
ht_res, hv_res = entrenar(m_resnet, dl_train, dl_val, "ResNet18")

print("\n=== Entrenando EfficientNet-B0 ===")
torch.manual_seed(SEMILLA)
ht_eff, hv_eff = entrenar(m_effnet, dl_train, dl_val, "EfficientNet-B0")


## 6. Comparación de curvas de pérdida

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

for ax, (nombre, ht, hv, color) in zip(axes, [
    ("ResNet18",       ht_res, hv_res, "#C44E52"),
    ("EfficientNet-B0", ht_eff, hv_eff, "#4C72B0"),
]):
    ax.plot(ht, color=color, linewidth=2, label="train")
    ax.plot(hv, color=color, linewidth=2, linestyle="--", label="val")
    ax.set_title(nombre, fontsize=12, fontweight="bold")
    ax.set_xlabel("Época"); ax.set_ylabel("Cross-Entropy Loss")
    ax.legend(); ax.grid(alpha=0.3)

plt.suptitle("Curvas de pérdida — ResNet18 vs EfficientNet-B0", fontsize=12)
plt.tight_layout()
plt.show()


## 7. Evaluación comparativa en test

In [ ]:
def evaluar(modelo, dl, nombre):
    modelo.eval()
    pred_all, real_all = [], []
    with torch.no_grad():
        for imgs, labels in dl:
            out  = modelo(imgs.to(device))
            pred = out.argmax(dim=1).cpu().numpy()
            pred_all.extend(pred)
            real_all.extend(labels.numpy())

    real_all = np.array(real_all)
    pred_all = np.array(pred_all)

    print(f"\n=== {nombre} ===")
    print(classification_report(real_all, pred_all,
                                 target_names=nombres_clases,
                                 zero_division=0))
    return real_all, pred_all

real_r, pred_r = evaluar(m_resnet, dl_test, "ResNet18")
real_e, pred_e = evaluar(m_effnet, dl_test, "EfficientNet-B0")


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

# Tabla comparativa
print("=== TABLA COMPARATIVA ===")
print(f"{'Arquitectura':<20} {'Accuracy':>10} {'F1 macro':>10} {'Parámetros':>12}")
print("-" * 55)
for nombre, real, pred, n_params in [
    ("ResNet18",        real_r, pred_r, "11.2 M"),
    ("EfficientNet-B0", real_e, pred_e, "5.3 M"),
]:
    acc = accuracy_score(real, pred)
    f1  = f1_score(real, pred, average="macro", zero_division=0)
    print(f"{nombre:<20} {acc:>10.3f} {f1:>10.3f} {n_params:>12}")

# Matrices de confusión lado a lado
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (nombre, real, pred) in zip(axes, [
    ("ResNet18",        real_r, pred_r),
    ("EfficientNet-B0", real_e, pred_e),
]):
    cm = confusion_matrix(real, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=nombres_clases)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    acc = accuracy_score(real, pred)
    f1  = f1_score(real, pred, average="macro", zero_division=0)
    ax.set_title(f"{nombre}\nAcc={acc:.3f} | F1={f1:.3f}")

plt.suptitle("Comparación de arquitecturas — matrices de confusión (test)",
             fontsize=12)
plt.tight_layout()
plt.show()


## 8. Decisión de arquitectura y justificación

**Arquitectura seleccionada para las etapas siguientes:** la que obtenga
mejor F1 macro en el test anterior.

**Justificación Feature Extraction vs Fine-Tuning:**
- Nuestro dataset tiene ~6.277 imágenes en un dominio muy distinto a ImageNet
  (matrices binarias abstractas vs fotografías naturales).
- Fine-Tuning profundo con pocos datos en dominio distinto arriesga destruir
  las representaciones preentrenadas útiles (detectores de bordes y texturas
  de bajo nivel que sí son transferibles).
- Feature Extraction es la estrategia estándar en este escenario
  (He et al., 2016; Tan & Le, 2019).


## 9. Grad-CAM — interpretabilidad del modelo ganador

In [ ]:
# Usar el modelo con mejor desempeño
# Cambiar 'mejor_modelo' y 'mejor_nombre' según los resultados de la sección 7
mejor_modelo = m_resnet   # ajustar si EfficientNet ganó
mejor_nombre  = "ResNet18"

def grad_cam(modelo, img_tensor, clase_objetivo=None):
    modelo.eval()
    activaciones, gradientes = [], []

    def hook_fwd(m, inp, out): activaciones.append(out.detach().clone())
    def hook_bwd(m, gin, gout):
        if gout[0] is not None: gradientes.append(gout[0].detach().clone())

    # Última capa conv de ResNet18 = layer4 / EfficientNet = features[-1]
    if hasattr(modelo, 'layer4'):
        target = modelo.layer4
    else:
        target = modelo.features[-1]

    hf = target.register_forward_hook(hook_fwd)
    hb = target.register_full_backward_hook(hook_bwd)

    for param in target.parameters():
        param.requires_grad_(True)

    img = img_tensor.unsqueeze(0).to(device)
    with torch.enable_grad():
        out = modelo(img)
        if clase_objetivo is None:
            clase_objetivo = out.argmax().item()
        modelo.zero_grad()
        out[0, clase_objetivo].backward(retain_graph=True)

    hf.remove(); hb.remove()
    for param in target.parameters():
        param.requires_grad_(False)

    if not gradientes:
        cam = activaciones[0].squeeze().mean(dim=0).cpu()
    else:
        grads = gradientes[0].squeeze().cpu()
        acts  = activaciones[0].squeeze().cpu()
        if grads.dim() == 3:
            pesos = grads.mean(dim=(1,2))
            cam   = torch.zeros(acts.shape[1:])
            for i, w in enumerate(pesos):
                cam += w * acts[i]
        else:
            cam = acts.mean(dim=0)

    cam = torch.relu(cam)
    if cam.max() > 0:
        cam = (cam - cam.min()) / (cam.max() - cam.min())
    return cam.cpu().numpy(), clase_objetivo

# Visualizar Grad-CAM para 2 ejemplos por clase
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
for c, nombre in enumerate(nombres_clases):
    indices_clase = [i for i, e in enumerate(etiquetas[idx_test]) if e == c][:2]
    for col, idx_local in enumerate(indices_clase):
        idx_global = idx_test[idx_local]
        ventana = retornos[idx_global:idx_global+WINDOW]
        mat = matriz_recurrencia(ventana)
        img_tensor = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((224,224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
        ])((np.stack([mat]*3, axis=-1)*255).astype(np.uint8))

        cam_map, pred = grad_cam(mejor_modelo, img_tensor, clase_objetivo=c)

        import torch.nn.functional as F
        cam_res = F.interpolate(
            torch.tensor(cam_map).unsqueeze(0).unsqueeze(0),
            size=(WINDOW, WINDOW), mode="bilinear", align_corners=False
        ).squeeze().numpy()

        col_idx = col * 2
        axes[c][col_idx].imshow(mat, cmap="binary", origin="upper")
        axes[c][col_idx].set_title(f"C{c}: {nombre}\nOriginal", fontsize=8)
        axes[c][col_idx].axis("off")

        axes[c][col_idx+1].imshow(mat, cmap="binary", origin="upper", alpha=0.5)
        axes[c][col_idx+1].imshow(cam_res, cmap="jet", origin="upper", alpha=0.6)
        axes[c][col_idx+1].set_title(f"C{c}: {nombre}\nGrad-CAM", fontsize=8)
        axes[c][col_idx+1].axis("off")

plt.suptitle(f"Grad-CAM — {mejor_nombre}\n"
             "(rojo = alta activación · azul = baja activación)", fontsize=11)
plt.tight_layout()
plt.show()


## 10. Guardar pesos de ambos modelos

In [ ]:
torch.save(m_resnet.state_dict(), "centinela_fase2_resnet18.pt")
torch.save(m_effnet.state_dict(),  "centinela_fase2_efficientnet_b0.pt")
print("Pesos guardados:")
print("  centinela_fase2_resnet18.pt")
print("  centinela_fase2_efficientnet_b0.pt")


## 11. Resumen de la Rama A

**Datos:** matrices de recurrencia desde TRM diaria (6.277 ventanas 30×30).
**Clases:** Depreciación (>+1% acum.) / Estable (±1%) / Apreciación (<-1%).

**Arquitecturas comparadas:**
- ResNet18 (11.2 M parámetros, 18 capas)
- EfficientNet-B0 (5.3 M parámetros, escalado compuesto, 82 capas)
- Estrategia: Feature Extraction en ambas (capas conv. congeladas).

**Data augmentation:** rotación ±15°, flip horizontal, variación de brillo ±20%.
Solo aplicada al conjunto de entrenamiento.

**Interpretabilidad:** Grad-CAM sobre el modelo ganador.

**Limitación honesta:** las representaciones de ImageNet no son óptimas
para matrices de recurrencia binarias. Fine-Tuning profundo o una CNN
entrenada desde cero podrían mejorar, pero requieren más datos.
